In [12]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib
from gene_thesaurus import GeneThesaurus

In [2]:
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-gex")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.gex.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

CALIBRATE_DATASET_PATH = os.path.join(DATASETS_PATH, "calibrate-gex")
X_calibrate_path = os.path.join(CALIBRATE_DATASET_PATH, "features.gex.csv")
y_calibrate_path = os.path.join(CALIBRATE_DATASET_PATH, "labels.csv")

TEST_DATASET_PATH = os.path.join(DATASETS_PATH, "test-gex")
X_test_path = os.path.join(TEST_DATASET_PATH, "features.gex.csv")
y_test_path = os.path.join(TEST_DATASET_PATH, "labels.csv")

output_dir = "/home/mariya/Data/pallas/models/gex"
random_seed = 42
rf_model_path = os.path.join(output_dir, "final_model.pkl")
selected_features_path = os.path.join(output_dir, "selected_features.txt")
pipeline_path = os.path.join(output_dir, "pipeline.pkl")
hyperparams_path = os.path.join(output_dir, "hyperparams.pkl")

# Print configuration
print(f"y_test_path: {y_test_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"rf_model_path: {rf_model_path}")
print(f"selected_features_path: {selected_features_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"hyperparams_path: {hyperparams_path}")

y_test_path: /home/mariya/Data/pallas/mlomix_datasets/test-gex/labels.csv
output_dir: /home/mariya/Data/pallas/models/gex
random_seed: 42
rf_model_path: /home/mariya/Data/pallas/models/gex/final_model.pkl
selected_features_path: /home/mariya/Data/pallas/models/gex/selected_features.txt
pipeline_path: /home/mariya/Data/pallas/models/gex/pipeline.pkl
hyperparams_path: /home/mariya/Data/pallas/models/gex/hyperparams.pkl


In [4]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, index_col=0)
X_cal = pd.read_csv(X_calibrate_path, index_col=0)
y_cal = pd.read_csv(y_calibrate_path, index_col=0)

# Append X_cal to X_test and y_cal to y_test
X_merged = pd.concat([X_cal, X_test], axis=0)
y_merged = pd.concat([y_cal, y_test], axis=0)
X_merged, y_merged

(                    ENSG00000000003  ENSG00000000971  ENSG00000001036  \
 id                                                                      
 16-003                     0.430054         3.011660         3.096152   
 16-012                     0.209022         0.592110         3.934643   
 16-014                     2.585009         1.734460         4.408511   
 16-017                    -0.034661         0.628366         3.471705   
 16-077                    -0.034661         0.758143         3.156128   
 ...                             ...              ...              ...   
 TARGET-21-PASLZE-D         0.013070         5.420226         4.998817   
 TCGA-AB-2913               0.013070         3.959354         6.605994   
 TCGA-AB-2919               0.013070         6.184728         4.068995   
 TCGA-AB-2924               0.013070         0.111694         5.456970   
 TCGA-AB-3011               0.013070         1.485269         4.777363   
 
                     ENSG00000001460

In [5]:
# align the indices
X_merged = X_merged.loc[y_merged.index]
X_merged, y_merged

(                    ENSG00000000003  ENSG00000000971  ENSG00000001036  \
 id                                                                      
 16-003                     0.430054         3.011660         3.096152   
 16-012                     0.209022         0.592110         3.934643   
 16-014                     2.585009         1.734460         4.408511   
 16-017                    -0.034661         0.628366         3.471705   
 16-077                    -0.034661         0.758143         3.156128   
 ...                             ...              ...              ...   
 TARGET-21-PASLZE-D         0.013070         5.420226         4.998817   
 TCGA-AB-2913               0.013070         3.959354         6.605994   
 TCGA-AB-2919               0.013070         6.184728         4.068995   
 TCGA-AB-2924               0.013070         0.111694         5.456970   
 TCGA-AB-3011               0.013070         1.485269         4.777363   
 
                     ENSG00000001460

In [6]:
print("\ny_test class distribution:")
print(y_merged.value_counts())


y_test class distribution:
class                   
B-ALL, high hyperdiploid    167
AML, KMT2A-r                156
B-ALL, ETV6::RUNX1          139
B-ALL, BCR::ABL1-like       112
AML, RUNX1::RUNX1T1         108
AML, NPM1alt                 68
B-ALL, KMT2A-r               66
B-ALL, PAX5alt               61
T-ALL                        53
AML, CBFB::MYH11             50
B-ALL, DUX4-r                49
B-ALL, BCR::ABL1             42
B-ALL, TCF3::PBX1            38
B-ALL, hypodiploid           37
AML, CEBPAalt                34
AML, NUP98-r                 29
B-ALL, ZNF384-r              24
B-ALL, ETV6::RUNX1-like      19
B-ALL, MEF2D-r               17
B-ALL, iAMP21                15
AML, DEK::NUP214             13
B-ALL, PAX5 P80R             11
AML, CBFA2T3::GLIS2          11
Non-leukemic                  7
AML, APL-PML::RARA            6
B-ALL, NUTM1-r                5
B-ALL, MYC-r                  5
AML, KAT6A-r                  4
AML, ETV6-r                   3
B-ALL, HLF-r       

In [7]:
y_merged = y_merged.iloc[:, 0].values # Convert to 1D numpy array
y_merged

array(['B-ALL, ETV6::RUNX1', 'B-ALL, ETV6::RUNX1',
       'B-ALL, high hyperdiploid', ..., 'AML, NPM1alt', 'AML, NPM1alt',
       'AML, NPM1alt'], shape=(1358,), dtype=object)

In [8]:
# Load model
final_rf = joblib.load(rf_model_path)

# Load selected features from text file
selected_features = pd.read_csv(selected_features_path, header=None)[0].tolist()

# Use selected features for test set
y_pred_final = final_rf.predict(X_merged)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_merged.shape}")

Number of selected features: 5884
Shape of X_test_selected: (1358, 5884)


In [9]:
print("\nTest set classification report (final optimized model):")
print(classification_report(y_merged, y_pred_final, digits=3))



Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      0.375     0.500     0.429         6
     AML, CBFA2T3::GLIS2      0.889     0.727     0.800        11
        AML, CBFB::MYH11      0.923     0.960     0.941        50
           AML, CEBPAalt      1.000     0.706     0.828        34
        AML, DEK::NUP214      0.579     0.846     0.688        13
             AML, ETV6-r      1.000     0.333     0.500         3
           AML, FUS::ERG      0.125     1.000     0.222         2
            AML, KAT6A-r      1.000     1.000     1.000         4
            AML, KMT2A-r      0.491     0.853     0.623       156
            AML, NPM1alt      0.707     0.603     0.651        68
            AML, NUP98-r      0.833     0.172     0.286        29
     AML, PICALM::MLLT10      0.000     0.000     0.000         2
     AML, RUNX1::RUNX1T1      0.798     0.880     0.837       108
        B-ALL, BCR

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [13]:
# ----------------------
# Extract top cpg sites from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Gene": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

gt = GeneThesaurus()
translated = gt.translate_genes(gene_importances["Gene"].tolist(),
                               source="ensembl_id", target="symbol")
gene_importances["Gene Symbol"] = gene_importances["Gene"].map(translated)

print("\nTop 20 genes driving classification (final model):")
print(gene_importances.head(20))


Top 20 genes driving classification (final model):
                 Gene  Importance Gene Symbol
4643  ENSG00000184507    0.028297       NUTM1
461   ENSG00000079102    0.026084     RUNX1T1
2637  ENSG00000145794    0.018835      MEGF10
5496  ENSG00000248329    0.018226       APELA
732   ENSG00000101115    0.017370       SALL4
1072  ENSG00000108511    0.011917       HOXB6
940   ENSG00000105642    0.011721       KCNN1
3082  ENSG00000157680    0.011458        DGKI
4672  ENSG00000185046    0.010392      ANKS1B
1484  ENSG00000120075    0.009492       HOXB5
3724  ENSG00000168004    0.009181      PLAAT5
3428  ENSG00000164100    0.008383       NDST3
1775  ENSG00000128510    0.008078        CPA4
5599  ENSG00000259803    0.007855    SLC22A31
2550  ENSG00000143995    0.007552       MEIS1
1637  ENSG00000124302    0.007545       CHST8
376   ENSG00000072657    0.007475       TRHDE
2535  ENSG00000143816    0.007365       WNT9A
847   ENSG00000103449    0.007070       SALL1
10    ENSG00000002745    0.0

In [14]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 25 out of 5884

Bottom 20 genes by importance (final model):
                 Gene  Importance Gene Symbol
2880  ENSG00000152208         0.0       GRID2
5658  ENSG00000266094         0.0      RASSF5
225   ENSG00000054938         0.0      CHRDL2
1925  ENSG00000131910         0.0       NR0B2
1826  ENSG00000130045         0.0       NXNL2
5630  ENSG00000262165         0.0   C17orf114
5645  ENSG00000264230         0.0     ANXA8L1
5216  ENSG00000204379         0.0      XAGE1A
4090  ENSG00000173548         0.0       SNX33
5502  ENSG00000248993         0.0         NaN
5499  ENSG00000248713         0.0     C4orf54
4415  ENSG00000180219         0.0      GARIN6
4867  ENSG00000188162         0.0        OTOG
3287  ENSG00000162592         0.0      CCDC27
2614  ENSG00000145386         0.0       CCNA2
726   ENSG00000101000         0.0       PROCR
4623  ENSG00000184276         0.0    DEFB108B
4009  ENSG00000172183         0.0       ISG20
4830  ENSG00000187627  